**ReadME**  
**This Version Does not contain any test Cases, Please refer to V1 or V2 for detailed Test Case Output**  
* V1: 25 Historical Quarters Data (5 Assets)
* V2: 100 Historial Quarters Data (5 Assets)

**Overview of the Entire JupyterNotebook**  
Section 1: Required Libraries  
Section 2: Data Processing
* 2.1 Single Asset Data Processing  
* 2.2 Single Asset Test Case  
* 2.3 Multi Asset Data Processing  
* 2.4 Multi Asset Test Case

Section 3: LSTM Machine Learning Estimator  
* 3.1 Pre-Processing
* 3.2 Pre-Processing Test Case
* 3.3 LSTM Model SetUp
* 3.4 LSTM Test Case

Section 4: ChatGPT 15 Asset Main Case

**Use Intruction**  
Please run all the sections with functions to construct the framework, Test Case Sections are optional to run

**Section 1: Required Libraries**

In [195]:
import pandas as pd
import yfinance as yf
import numpy as np

from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Dense
from tensorflow.keras.regularizers import l2

**Section 2: Data Processing**  
Section 2.1 Single Asset Data Processing

In [241]:
def get_historical_returns(ticker, start_date, end_date, frequency="monthly"):
    'Function to fetch Historical Price data and compute returns'

    data = yf.download(ticker,start=start_date, end=end_date)

    # Calculate Daily Returns
    daily_data = data.copy()
    daily_data['Return'] = daily_data['Close'].pct_change()
    daily_returns = daily_data[['Return']].dropna()

    # Calculate Monthly Returns
    monthly_data = data.copy()
    monthly_data['Return'] = monthly_data['Close']
    monthly_data = monthly_data['Return'].resample('M').last()
    monthly_returns = monthly_data.pct_change() 
    monthly_returns = monthly_returns.dropna()
    
    if frequency == "daily": return daily_returns
    if frequency == "monthly": return monthly_returns

    return monthly_data

def resample_quaterly_data(quaterly_data, target_data):
    'Repeat the quaterly available ratios to same frequency as target return'
    
    quaterly_data.index = pd.to_datetime(quaterly_data.index)
    target_data.index = pd.to_datetime(target_data.index)

    # Resample the quaterly data to daily frequency using Forward Fill
    quaterly_data.index = quaterly_data.index + pd.DateOffset(days=1)
    aligned_quaterly_data = quaterly_data.reindex(target_data.index, method='ffill')

    aligned_quaterly_data = aligned_quaterly_data.dropna()
    return aligned_quaterly_data


def load_features(path_to_file, ticker, start_date, end_date):
    'Function to Load all features for a single company'

    # Load the Excel file and read Data from the file
    file_path = path_to_file + ticker + '.xlsx'
    sheet_name = ticker + '-US'           
    data = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')

    # Remove rows with any NaN values
    # Because time frame is longer, cannot apply this
    # data = data.dropna()

    # Reset the index of the DataFrame and drop the old index
    data = data.reset_index(drop=True)

    data = data.set_index('Date').T
    data.index = pd.to_datetime(data.index, format='%b \'%y')
    data.index = data.index + pd.offsets.MonthEnd()
    ratio_data = data.apply(pd.to_numeric)

    # Select a few columns
    pe_column = 'Price/Earnings'  
    pb_column = 'Price/Book Value'  
    roa_column = 'Return on Assets' 
    roe_column = 'Return on Equity ' 
    fcf_column = 'Free Cash Flow per Share'
    ratio_data = data[[pe_column, pb_column, roa_column, roe_column, fcf_column]]
    
    # Drop N/A dates
    # Removing rows with any NaN values
    ratio_data = ratio_data.dropna()

    # Process Return Data
    returns_data = get_historical_returns(ticker, start_date, end_date)
    adjusted_ratio_data = resample_quaterly_data(ratio_data, returns_data)
    features = pd.concat([adjusted_ratio_data, returns_data],axis=1)

    return features

Section 2.3 Multi Asset Data Processing

In [197]:
def multi_df(path_to_file, ticker_list, start_date, end_date):
    company_data = {}
    for ticker in ticker_list:
        company_data[ticker] = load_features(path_to_file, ticker, start_date, end_date)

    # Initialize a list to hold DataFrames with the new multi-index
    multi_index_dfs = []

    for company, df in company_data.items():
        # Set the company name as an additional level in the index
        df_multi_index = df.copy()
        df_multi_index['Company'] = company
        df_multi_index.set_index(['Company', df_multi_index.index], inplace=True)
        
        # Append to the list
        multi_index_dfs.append(df_multi_index)

    # Concatenate all DataFrames into a single multi-index DataFrame
    final_df = pd.concat(multi_index_dfs)

    return final_df

**Section 3: LSTM Machine Learning Estimator**  
Section 3.1 Pre-Processing

In [198]:
def create_sequences(features, targets, seq_length):
    'Function to create sequence'
    'Need to define the sequence length: e.g. using 4 quaters to predict the next quater'

    xs = []
    ys = []

    for i in range(len(features)-seq_length):
        x = features[i:(i+seq_length)]
        y = targets.iloc[i+seq_length]
        xs.append(x)
        ys.append(y)
        
    return np.array(xs), np.array(ys)

Section 3.3 LSTM Model SetUp

In [199]:
# LSTM Model Set Up

# Model architecture
model = tf.keras.Sequential([
    LSTM(512, return_sequences=True),
    Dropout(0.02),
    LSTM(256, return_sequences=True),
    LSTM(128),
    Dense(1, activation='linear', kernel_regularizer=l2(0.0005))
])

# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)
model.compile(optimizer=optimizer, loss='mean_squared_error')

ChatGPT 15 Stock Test Case

Step 1: Define Input and Parameters

In [200]:
# 1. File Path
path_to_file = ''
# 2. Ticker List
ChatGPT_stocks = ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'JNJ', 'V', 'PG', 'JPM', 'UNH', 'MA', 'INTC', 'VZ', 'GOOG', 'HD', 'T']
# 3. Target Time Frame
start_date = '2016-09-30'
end_date = '2021-09-30'
# 4. Sequence Length
seq_length = 6
# 5. Training and Validation Set Split Ratio
train_ratio = 0.8
# 6. Num Epoch and Num Batch
num_epoch = 20
num_batch = 12

Step2: Pre-Processing

In [201]:
# Loading Phase: Took a while to run this (Don't Rerun)
final_df = multi_df(path_to_file, ChatGPT_stocks, start_date, end_date)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%*******

In [202]:
multi_df_gpt = final_df
multi_return_gpt = pd.DataFrame(multi_df_gpt['Return'])
# print(multi_df_gpt)
# print(multi_return_gpt)

In [203]:
features = multi_df_gpt
targets = multi_return_gpt
multi_X_gpt, multi_y_gpt = create_sequences(features, targets, seq_length)
# print(multi_X_test1)
# print(multi_y_test1)

Step3: Train Model on Training and Validation Sets

In [242]:
# Variable in Use and Constant Define
X = multi_X_gpt
y = multi_y_gpt

# Split data into training and validation sets
train_size = int(len(X) * train_ratio)
X_train, X_vali = X[:train_size], X[train_size:]
y_train, y_vali = y[:train_size], y[train_size:]

# Train the model
model.fit(X_train, y_train, epochs=num_epoch, batch_size=num_batch, validation_data=(X_vali, y_vali))

Epoch 1/20
60/60 [==============================] - 1s 15ms/step - loss: 0.0046 - val_loss: 0.0044
Epoch 2/20
60/60 [==============================] - 1s 15ms/step - loss: 0.0047 - val_loss: 0.0042
Epoch 3/20
60/60 [==============================] - 1s 14ms/step - loss: 0.0046 - val_loss: 0.0042
Epoch 4/20
60/60 [==============================] - 1s 14ms/step - loss: 0.0047 - val_loss: 0.0042
Epoch 5/20
60/60 [==============================] - 1s 14ms/step - loss: 0.0046 - val_loss: 0.0042
Epoch 6/20
60/60 [==============================] - 1s 14ms/step - loss: 0.0046 - val_loss: 0.0041
Epoch 7/20
60/60 [==============================] - 1s 14ms/step - loss: 0.0046 - val_loss: 0.0041
Epoch 8/20
60/60 [==============================] - 1s 14ms/step - loss: 0.0045 - val_loss: 0.0041
Epoch 9/20
60/60 [==============================] - 1s 14ms/step - loss: 0.0045 - val_loss: 0.0041
Epoch 10/20
60/60 [==============================] - 1s 14ms/step - loss: 0.0045 - val_loss: 0.0043
Epoch 11/

In [243]:
# Check Trianing Set Output
X_pred, y_pred = X_train, y_train

predictions = model.predict(X_pred)
print(predictions)

 1/23 [>.............................] - ETA: 0s

23/23 [==============================] - 0s 6ms/step
[[0.0343072 ]
 [0.03448775]
 [0.03440532]
 [0.03465694]
 [0.03475471]
 [0.03458744]
 [0.03491929]
 [0.03505403]
 [0.03518666]
 [0.03531688]
 [0.03589467]
 [0.03589483]
 [0.03620013]
 [0.03766683]
 [0.03767941]
 [0.03803803]
 [0.03972144]
 [0.03972097]
 [0.0400693 ]
 [0.03613043]
 [0.03587337]
 [0.03565791]
 [0.03776319]
 [0.0380832 ]
 [0.03809293]
 [0.03916353]
 [0.03959403]
 [0.03936561]
 [0.04017227]
 [0.04035804]
 [0.04032277]
 [0.04192941]
 [0.04208681]
 [0.04211416]
 [0.04173794]
 [0.04179804]
 [0.04174618]
 [0.04300907]
 [0.04312067]
 [0.04314339]
 [0.04339088]
 [0.04300287]
 [0.04253308]
 [0.04201633]
 [0.04156677]
 [0.04124644]
 [0.04060944]
 [0.04076659]
 [0.04089393]
 [0.04043385]
 [0.04044893]
 [0.04038633]
 [0.03975224]
 [0.03956642]
 [0.03946607]
 [0.03736319]
 [0.03745843]
 [0.0369162 ]
 [0.03653868]
 [0.03549913]
 [0.03386024]
 [0.03307641]
 [0.0333923 ]
 [0.03403182]
 [0.03478735]
 [0.03523578]
 [0.03557824]
 [0.03816

In [244]:
print(X_train)

[[[ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00  4.33434631e-03]
  [ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00 -2.65985930e-02]
  [ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00  4.79551503e-02]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    2.52979700e+00  4.77464928e-02]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    2.52979700e+00  1.28883455e-01]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    2.52979700e+00  4.86896701e-02]]

 [[ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00 -2.65985930e-02]
  [ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00  4.79551503e-02]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    2.52979700e+00  4.77464928e-02]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    

Step4: Evaluate the Model on Test Set

In [351]:
# Test Period 1: September 30 2021 to July 30 2023
# Test Period 2: March 14, 2023 to July 31 2023
# Test Period 3: May 01 2023 to July 31 2023

start_t1 = '2021-09-30'
end_t1 = '2023-07-31'

start_t2 = '2023-03-14'
end_t2 = '2023-07-31'

start_t3 = '2023-05-01'
end_t3 = '2023-07-31'

In [352]:
# Loading Test Phase: Took a while to run this (Don't Rerun)
final_df_t1 = multi_df(path_to_file, ChatGPT_stocks, start_t1, end_t1)
# final_df_t2 = multi_df(path_to_file, ChatGPT_stocks, start_t2, end_t2)
# final_df_t3 = multi_df(path_to_file, ChatGPT_stocks, start_t3, end_t3)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%*******

In [353]:
multi_df_t1 = final_df_t1
multi_return_t1 = pd.DataFrame(multi_df_t1['Return'])
seq_length = 6
multi_X_t1, multi_y_t1 = create_sequences(multi_df_t1, multi_return_t1, seq_length)

# multi_df_t2 = final_df_t2
# multi_return_t2 = pd.DataFrame(multi_df_t2['Return'])
# seq_length = 4
# multi_X_t2, multi_y_t2 = create_sequences(multi_df_t2, multi_return_t2, seq_length)

# multi_df_t3 = final_df_t3
# multi_return_t3 = pd.DataFrame(multi_df_t3['Return'])
# seq_length = 2
# multi_X_t3, multi_y_t3 = create_sequences(multi_df_t3, multi_return_t3, seq_length)

In [354]:
def generate_sequence_mapping(df,freq):
    sequences = []
    sequence_mappings = []

    for company in df.index.get_level_values(0).unique():
        # Get the data for the current company
        company_data = df.xs(company, level='Company')
        
        # Create 6-month sequences and record their mappings
        for i in range(len(company_data) - (freq-1)):
            sequence = company_data.iloc[i:i+freq]
            if sequence.shape[0] == freq:  # Ensure each sequence has 6 months
                sequences.append(sequence.drop(columns='Return').values)  # Add the sequence to the list, excluding 'Return' if it's not an input feature
                end_date = sequence.index[-1]  # The end date of the sequence
                sequence_mappings.append((company, end_date))  # Record the mapping
    return sequence_mappings


**Evaluation for Each test Period Execution parts**

In [355]:
# Change here to change test time frame (t1/t2/t3)
# Test Period 1: September 30 2021 to July 30 2023
# Test Period 2: March 14, 2023 to July 31 2023
# Test Period 3: May 01 2023 to July 31 2023
df = multi_df_t1
X_test, y_test = multi_X_t1, multi_y_t1

# Parse accordingly before feeding into the model
sequence_mappings = generate_sequence_mapping(df,6)
print(sequence_mappings)

[('AAPL', Timestamp('2022-03-31 00:00:00')), ('AAPL', Timestamp('2022-04-30 00:00:00')), ('AAPL', Timestamp('2022-05-31 00:00:00')), ('AAPL', Timestamp('2022-06-30 00:00:00')), ('AAPL', Timestamp('2022-07-31 00:00:00')), ('AAPL', Timestamp('2022-08-31 00:00:00')), ('AAPL', Timestamp('2022-09-30 00:00:00')), ('AAPL', Timestamp('2022-10-31 00:00:00')), ('AAPL', Timestamp('2022-11-30 00:00:00')), ('AAPL', Timestamp('2022-12-31 00:00:00')), ('AAPL', Timestamp('2023-01-31 00:00:00')), ('AAPL', Timestamp('2023-02-28 00:00:00')), ('AAPL', Timestamp('2023-03-31 00:00:00')), ('AAPL', Timestamp('2023-04-30 00:00:00')), ('AAPL', Timestamp('2023-05-31 00:00:00')), ('AAPL', Timestamp('2023-06-30 00:00:00')), ('AAPL', Timestamp('2023-07-31 00:00:00')), ('MSFT', Timestamp('2022-03-31 00:00:00')), ('MSFT', Timestamp('2022-04-30 00:00:00')), ('MSFT', Timestamp('2022-05-31 00:00:00')), ('MSFT', Timestamp('2022-06-30 00:00:00')), ('MSFT', Timestamp('2022-07-31 00:00:00')), ('MSFT', Timestamp('2022-08-31 

In [357]:
# Evaluate the model
loss = model.evaluate(X_test, y_test)
print(f"Mean Squared Error: {loss}")

11/11 [==============================] - 0s 6ms/step - loss: 0.0087
Mean Squared Error: 0.008704284206032753


In [358]:
# Make predictions
predictions = model.predict(X_test)

11/11 [==============================] - 0s 7ms/step


In [360]:
# Prediction Result Mapping back to Each Company
results_list = []

# Use the shorter length of the two lists to avoid IndexError
min_length = min(len(predictions), len(sequence_mappings))

for i in range(min_length):
    company, end_date = sequence_mappings[i]
    expected_return = predictions[i][0]
    results_list.append({'Company': company, 'EndDate': end_date, 'ExpectedReturn': expected_return})

results_df = pd.DataFrame(results_list)

In [363]:
# Generate Covariance Matrix

# Pivot the DataFrame so that each company's returns form a column
pivoted_df = results_df.pivot(index='EndDate', columns='Company', values='ExpectedReturn')
pivoted_df.isna().any()


# Calculate the covariance matrix
# covariance_matrix now contains the covariance of returns between companies
covariance_matrix = pivoted_df.cov()
print(covariance_matrix)

Company          AAPL          AMZN          GOOG         GOOGL            HD  \
Company                                                                         
AAPL     1.128652e-07 -2.937577e-07 -2.773230e-07 -6.831851e-07  1.242850e-07   
AMZN    -2.937577e-07  3.881243e-06  6.525789e-07  3.762812e-07 -3.911693e-07   
GOOG    -2.773230e-07  6.525789e-07  2.361323e-05  1.244616e-05 -4.222177e-06   
GOOGL   -6.831851e-07  3.762812e-07  1.244616e-05  1.296262e-05 -2.752029e-06   
HD       1.242850e-07 -3.911693e-07 -4.222177e-06 -2.752029e-06  9.118677e-07   
INTC    -1.294737e-07  8.565708e-07  8.318473e-07  1.434026e-07 -2.231199e-07   
JNJ      1.148041e-07  2.909071e-07  3.039921e-08 -5.988813e-07  4.201415e-08   
JPM     -2.255126e-07  1.089068e-06  9.178177e-06  5.799009e-06 -1.853705e-06   
MA       3.750380e-08 -3.641638e-07 -1.241851e-06 -8.323406e-07  2.811366e-07   
MSFT    -8.528421e-08  5.257091e-07  2.089374e-06  1.293873e-06 -4.687919e-07   
PG       1.983200e-07 -2.697

In [365]:
# Ouput the final Average Expected return for each company
# Group by 'Company' and calculate the mean of 'ExpectedReturn'
# average_returns now contains the average expected return for each company
average_returns = results_df.groupby('Company')['ExpectedReturn'].mean()
print(average_returns)

Company
AAPL     0.038909
AMZN     0.038737
GOOG     0.034923
GOOGL    0.038973
HD       0.026819
INTC     0.037013
JNJ      0.034375
JPM      0.039632
MA       0.032147
MSFT     0.041827
PG       0.037970
T        0.025272
UNH      0.036909
V        0.033919
VZ       0.037922
Name: ExpectedReturn, dtype: float32


**For Test Period 2: March 14, 2023 to July 31 2023**


In [366]:
results_df['EndDate'] = pd.to_datetime(results_df['EndDate'])
filtered_df_t2 = results_df[results_df['EndDate'] >= '2023-03-14']
filtered_df_t2.isna().any()

Company           False
EndDate           False
ExpectedReturn    False
dtype: bool

In [367]:
# Generate Covariance Matrix

# Pivot the DataFrame so that each company's returns form a column
pivoted_df = filtered_df_t2.pivot(index='EndDate', columns='Company', values='ExpectedReturn')

# Calculate the covariance matrix
# covariance_matrix now contains the covariance of returns between companies
covariance_matrix = pivoted_df.cov()
print(covariance_matrix)

Company          AAPL          AMZN          GOOG         GOOGL            HD  \
Company                                                                         
AAPL     1.739318e-07 -5.182218e-07  3.932547e-07 -1.005151e-07 -5.654428e-08   
AMZN    -5.182218e-07  1.831842e-06 -9.488474e-07  2.380628e-08  1.558280e-07   
GOOG     3.932547e-07 -9.488474e-07  1.067373e-06 -4.587434e-07 -1.365663e-07   
GOOGL   -1.005151e-07  2.380628e-08 -4.587434e-07  3.876800e-07  4.690619e-08   
HD      -5.654428e-08  1.558280e-07 -1.365663e-07  4.690619e-08  2.267916e-08   
INTC    -1.866082e-08  1.943335e-07  6.908299e-08 -1.283070e-07  4.563094e-09   
JNJ      2.351183e-07 -8.264236e-07  4.595456e-07 -9.980530e-08 -6.885151e-08   
JPM      1.176400e-07 -1.352766e-07  4.267712e-07 -2.521066e-07 -4.649140e-08   
MA      -4.070307e-08  1.078949e-07 -1.034348e-07  4.246806e-08  1.544156e-08   
MSFT    -2.209732e-08  1.000097e-07 -3.123791e-08  1.013522e-08  7.888267e-09   
PG      -1.836324e-07  4.906

In [368]:
# Ouput the final Average Expected return for each company
# Group by 'Company' and calculate the mean of 'ExpectedReturn'
# average_returns now contains the average expected return for each company
average_returns = filtered_df_t2.groupby('Company')['ExpectedReturn'].mean()
print(average_returns)

Company
AAPL     0.038933
AMZN     0.038912
GOOG     0.028264
GOOGL    0.035776
HD       0.028008
INTC     0.036831
JNJ      0.034364
JPM      0.037063
MA       0.032470
MSFT     0.041248
PG       0.040811
T        0.024917
UNH      0.036568
V        0.033762
VZ       0.038472
Name: ExpectedReturn, dtype: float32


**For Test Period 3: Test Period 3: May 01 2023 to July 31 2023**


In [369]:
results_df['EndDate'] = pd.to_datetime(results_df['EndDate'])
filtered_df_t3 = results_df[results_df['EndDate'] >= '2023-05-01']

In [370]:
# Generate Covariance Matrix

# Pivot the DataFrame so that each company's returns form a column
pivoted_df = filtered_df_t3.pivot(index='EndDate', columns='Company', values='ExpectedReturn')

# Calculate the covariance matrix
# covariance_matrix now contains the covariance of returns between companies
covariance_matrix = pivoted_df.cov()
print(covariance_matrix)

Company          AAPL          AMZN          GOOG         GOOGL            HD  \
Company                                                                         
AAPL     1.065973e-07 -5.057402e-07  9.690619e-08  1.281226e-07 -1.679091e-08   
AMZN    -5.057402e-07  2.399940e-06 -4.569643e-07 -6.161363e-07  7.996738e-08   
GOOG     9.690619e-08 -4.569643e-07  1.034902e-07  7.095245e-08 -1.358795e-08   
GOOGL    1.281226e-07 -6.161363e-07  7.095245e-08  2.886088e-07 -2.513896e-08   
HD      -1.679091e-08  7.996738e-08 -1.358795e-08 -2.513896e-08  2.827425e-09   
INTC    -9.090426e-08  4.330236e-07 -7.307656e-08 -1.375403e-07  1.536044e-08   
JNJ      2.262155e-07 -1.062688e-06  2.638107e-07  9.990559e-08 -2.929896e-08   
JPM     -6.413437e-08  3.021222e-07 -7.017299e-08 -4.198610e-08  8.809675e-09   
MA      -1.170946e-08  5.502671e-08 -1.354869e-08 -5.487164e-09  1.528215e-09   
MSFT    -2.838763e-08  1.313629e-07 -4.407409e-08  1.989856e-08  2.482198e-09   
PG      -7.600401e-08  3.579

In [371]:
# Ouput the final Average Expected return for each company
# Group by 'Company' and calculate the mean of 'ExpectedReturn'
# average_returns now contains the average expected return for each company
average_returns = filtered_df_t3.groupby('Company')['ExpectedReturn'].mean()
print(average_returns)

Company
AAPL     0.038686
AMZN     0.039492
GOOG     0.027588
GOOGL    0.036089
HD       0.028115
INTC     0.036805
JNJ      0.034094
JPM      0.036790
MA       0.032544
MSFT     0.041278
PG       0.041103
T        0.024915
UNH      0.035748
V        0.033723
VZ       0.038536
Name: ExpectedReturn, dtype: float32
